In [12]:
import os

os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch
import numpy as np
import src.tools as tools

from importlib import reload

path_to_file = keras.utils.get_file(
        'shakespeare.txt',
        'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
        )
print(path_to_file)

with open(path_to_file) as f:
    shakespeare_text = f.read()
print(shakespeare_text[:500])

characters = sorted(set(shakespeare_text))
char_to_id = {char: index for index, char in enumerate(characters)}

shakespeare_tensor = torch.tensor(
        [char_to_id[char] for char in shakespeare_text],
        dtype=torch.long,
        )

print(shakespeare_tensor.shape)  # (text_length,)

/Users/thomasbayer/.keras/datasets/shakespeare.txt
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor
torch.Size([1115394])


In [6]:
vocab = list(set(shakespeare_text.lower().strip()))
print(vocab)
print(f"vocab size = {len(vocab)}")

[':', 'x', '&', '?', 'w', 'p', 'o', 'f', '!', 'a', '$', 'r', 'b', 'v', 'q', '-', 'd', ';', 'l', 'e', 'g', 'y', 'n', "'", '3', 's', ' ', 'm', 'k', 'u', 't', '\n', 'i', 'c', 'h', '.', 'j', 'z', ',']
vocab size = 39


In [18]:
tokenizer = tools.CharacterTokenizer(vocab)
encoded_text = tokenizer.encode(shakespeare_text)
print(f"Encoded text: {encoded_text[:100]}")

Encoded text: tensor([19, 22, 31, 32, 33,  2, 16, 22, 33, 22, 39, 18, 27, 11,  1, 15, 18, 19,
        28, 31, 18,  2, 36, 18,  2, 29, 31, 28, 16, 18, 18, 17,  2, 14, 27, 38,
         2, 19, 34, 31, 33, 21, 18, 31,  7,  2, 21, 18, 14, 31,  2, 26, 18,  2,
        32, 29, 18, 14, 24,  9,  1,  1, 14, 25, 25, 11,  1, 32, 29, 18, 14, 24,
         7,  2, 32, 29, 18, 14, 24,  9,  1,  1, 19, 22, 31, 32, 33,  2, 16, 22,
        33, 22, 39, 18, 27, 11,  1, 38, 28, 34])


In [19]:
reload(tools)

n_steps = 50
batch_size = 64
train_dataset = tools.text_to_dataset(
        encoded, n_tokens, n_steps=n_steps, batch_size=batch_size
        )

X, y = next(iter(train_dataset))
print(f"X = {X.shape}")  # (64, 50, n_tokens)
print(f"y = {y.shape}")  # (64, 50)

X = torch.Size([64, 50, 40])
y = torch.Size([64, 50])


In [35]:
m1 = keras.Sequential(
        [
                keras.layers.Input(shape=(None, n_tokens)),
                keras.layers.GRU(128, return_sequences=True),
                keras.layers.Dense(n_tokens, activation="softmax")
                ],
        )
m1.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
m1.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_3 (GRU)                     │ (None, None, 128)      │        65,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, None, 40)       │         5,160 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 70,440 (275.16 KB)

 Trainable params: 70,440 (275.16 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
import torch.nn.functional as F

model = m1
text = "to be or no"
tokens = tokenizer.encode(text)
print(tokens)
input = F.one_hot(tokens, num_classes=n_tokens).float()
print(input[0])
print(input.shape)
input = input.unsqueeze(0)  # Add batch dimension
print(input.shape)

tensor([33, 28,  2, 15, 18,  2, 28, 31,  2, 27, 28])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
        0., 0., 0., 0.])
torch.Size([11, 40])
torch.Size([1, 11, 40])


In [25]:
prediction = model(input)
print(prediction.shape)

torch.Size([1, 11, 40])


In [28]:
print(prediction[0, :, :].argmax(axis=-1))
next_token = prediction[0, -1, :].argmax(axis=-1)
print(next_token)

tensor([22, 26, 26,  9,  9, 19, 23, 24, 24, 12, 26], device='mps:0')
tensor(26, device='mps:0')


In [29]:
tokenizer.decode(next_token)

'm'

In [36]:
early_stopping = keras.callbacks.EarlyStopping(monitor="loss", min_delta=0.03, patience=2)
history = m1.fit(
        train_dataset,
        epochs=10,
        shuffle=False,  # The DataLoader controls shuffling.
        callbacks=[early_stopping],
        verbose=1
        )
m1.save("../../models/shakespeare_gru_1.keras")

Epoch 1/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 429s 25ms/step - loss: 1.7857
Epoch 2/10
  744/17428 ━━━━━━━━━━━━━━━━━━━━ 6:46 24ms/step - loss: 1.6649

KeyboardInterrupt: 

In [40]:
m1 = keras.models.load_model("../models/shakespeare_gru_1.keras")
prediction = m1(input)
print(prediction.shape)
print(prediction[0, :, :].argmax(axis=-1))
next_token = prediction[0, -1, :].argmax(axis=-1)
tokenizer.decode(next_token)

torch.Size([1, 11, 40])
tensor([21,  2, 21, 18,  2, 32, 27,  2, 33, 28, 33], device='mps:0')


't'

In [44]:
reload(tools)

tools.generate_text(m1, tokenizer, "to be", n_next=30)

'to be so make your father and the s'

In [42]:
m2 = keras.Sequential(
        [
                keras.layers.Input(shape=(None, n_tokens)),
                keras.layers.GRU(64, return_sequences=True),
                keras.layers.GRU(64, return_sequences=True),
                keras.layers.Dense(n_tokens, activation="softmax")
                ],
        )
m2.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
m2.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_4 (GRU)                     │ (None, None, 64)       │        20,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_5 (GRU)                     │ (None, None, 64)       │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, None, 40)       │         2,600 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,912 (187.16 KB)

 Trainable params: 47,912 (187.16 KB)

 Non-trainable params: 0 (0.00 B)

In [43]:
generate_text(m2, tokenizer, "to be", n_next=30)

'to beghhtk:jwwlmlo glq.pjzzzzhtzhhk'